In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load CSVs
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_err = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (unix seconds -> UTC)
for df in (df_metric, df_trace, df_log, df_err):
    df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Incident window (UTC)
window_start = pd.to_datetime("2024-01-24 04:08:41", utc=True)
window_end = pd.to_datetime("2024-01-24 04:38:41", utc=True)

# Helper to aggregate metric-like files (group_cols, name_col)
def aggregate_timeseries(df, group_cols, value_col='value'):
    # Global aggregates (must be computed on full series)
    agg_total = (
        df.groupby(group_cols)[value_col]
          .agg(count_total='size',
               p50_value=lambda x: x.quantile(0.5),
               p95_value=lambda x: x.quantile(0.95))
          .reset_index()
    )
    # Windowed aggregates
    df_win = df[(df['ts'] >= window_start) & (df['ts'] <= window_end)]
    agg_win = (
        df_win.groupby(group_cols)['ts']
              .agg(count_in_window='size',
                   first_ts_in_window='min',
                   last_ts_in_window='max')
              .reset_index()
    )
    # Merge and fill missing window info
    agg = pd.merge(agg_total, agg_win, on=group_cols, how='left')
    agg['count_in_window'] = agg['count_in_window'].fillna(0).astype(int)
    # Format timestamps compactly (ISO without fractional seconds)
    agg['first_ts_in_window'] = agg['first_ts_in_window'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    agg['last_ts_in_window'] = agg['last_ts_in_window'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    # Replace 'NaT' strings with empty (more compact)
    agg['first_ts_in_window'] = agg['first_ts_in_window'].replace('NaT', pd.NA)
    agg['last_ts_in_window'] = agg['last_ts_in_window'].replace('NaT', pd.NA)
    # Order columns as requested
    out_cols = group_cols + ['count_total', 'p50_value', 'p95_value',
                             'count_in_window', 'first_ts_in_window', 'last_ts_in_window']
    agg = agg[out_cols]
    # Sort by total count desc and limit to top 20
    agg = agg.sort_values('count_total', ascending=False).head(20).reset_index(drop=True)
    return agg

# Metric aggregation: group by (cmdb_id, kpi_name)
metric_res = aggregate_timeseries(df_metric, ['cmdb_id', 'kpi_name'])

# Trace aggregation: group by (cmdb_id, trace_name)
trace_res = aggregate_timeseries(df_trace, ['cmdb_id', 'trace_name'])

# Log aggregation: group by (cmdb_id, log_name)
log_res = aggregate_timeseries(df_log, ['cmdb_id', 'log_name'])

# Error logs: group by cmdb_id (no numeric 'value' for p50/p95)
# Total counts
err_total = df_err.groupby('cmdb_id').agg(count_total=('message','size')).reset_index()
# Windowed counts and first/last ts
df_err_win = df_err[(df_err['ts'] >= window_start) & (df_err['ts'] <= window_end)]
err_win = df_err_win.groupby('cmdb_id').agg(
    count_in_window=('ts','size'),
    first_ts_in_window=('ts','min'),
    last_ts_in_window=('ts','max')
).reset_index()
err_agg = pd.merge(err_total, err_win, on='cmdb_id', how='left')
err_agg['count_in_window'] = err_agg['count_in_window'].fillna(0).astype(int)
err_agg['first_ts_in_window'] = err_agg['first_ts_in_window'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
err_agg['last_ts_in_window'] = err_agg['last_ts_in_window'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
err_agg['first_ts_in_window'] = err_agg['first_ts_in_window'].replace('NaT', pd.NA)
err_agg['last_ts_in_window'] = err_agg['last_ts_in_window'].replace('NaT', pd.NA)
err_res = err_agg.sort_values('count_total', ascending=False).head(20).reset_index(drop=True)

# Compact list of unique cmdb_id values across all four files
unique_ids = sorted(set(
    df_metric['cmdb_id'].dropna().unique().tolist() +
    df_trace['cmdb_id'].dropna().unique().tolist() +
    df_log['cmdb_id'].dropna().unique().tolist() +
    df_err['cmdb_id'].dropna().unique().tolist()
))
unique_cmdb = pd.Series(unique_ids, name='cmdb_id')

# Display aggregated results (compact)
metric_res, trace_res, log_res, err_res, unique_cmdb

```
Out[1]:
```
Summary of the aggregated telemetry (incident window: 2024-01-24 04:08:41 → 2024-01-24 04:38:41 UTC)

Key points
- All top metric/trace/log groups have full coverage inside the incident window: for the displayed top rows count_total = 25 and count_in_window = 25, with first sample at 2024-01-24T04:11:00Z and last at 2024-01-24T04:35:00Z.
- metric.csv (top groups by total rows):
  - ts-voucher-service / kpi socket — count 25, p50=4, p95=4.
  - ts-admin-basic-info-service — multiple KPIs (cpu, diskio, latency-50/90, mem, socket, workload) all count 25 with p50/p95 shown in the table.
  - ts-admin-order-service and ts-admin-route-service — cpu/diskio/mem/socket KPIs all present (count 25).
- trace.csv (top groups):
  - ts-travel2-service.trace.to_ts-travel2-service.row_count — count 25, p50=607, p95=658.
  - Several root->service trace entries (durations, p95, error_rate, row_count) for admin-basic-info, admin-travel, assurance, auth, ticketinfo, etc., all count 25 with p50/p95 values in the table.
- log.csv (top groups):
  - Many services have log.row_count and log.error_count entries (e.g., ts-auth-service row_count p50=358, ts-travel-service row_count p50=451, ts-travel2-service row_count p50=504). error_count is zero (p50/p95 = 0) for the top rows.
- error_logs.csv:
  - No aggregated rows returned (empty) — there were no error_log groups counted in the result (no error log entries contributing to the top results in the window).
- Coverage:
  - Unique cmdb_id values present across the four files: 69 distinct services/components.

Interpretation / concise takeaway
- The sampled telemetry is complete across the incident window for the top groups (every top group recorded samples throughout the window).
- There are no error_log entries returned in the aggregation (no error logs captured for the top groups in the window).
- Metric and trace data show activity across many services (notably admin-* and travel-* services, and voucher/auth), with specific KPIs (socket, cpu, diskio, mem, latency, row_count) available and summarized by p50/p95 in the tables.

The original code execution output of IPython Kernel is also provided below for reference:

(                        cmdb_id    kpi_name  count_total     p50_value     p95_value  count_in_window    first_ts_in_window     last_ts_in_window
0            ts-voucher-service      socket           25  4.000000e+00  4.000000e+00               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
1   ts-admin-basic-info-service         cpu           25  3.403382e-01  6.742782e-01               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
2   ts-admin-basic-info-service      diskio           25  5.156425e+04  5.940996e+04               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
3   ts-admin-basic-info-service  latency-50           25  1.750000e-02  1.805110e-02               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
4   ts-admin-basic-info-service  latency-90           25  2.350000e-02  2.526702e-02               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
5   ts-admin-basic-info-service         mem           25  2.331173e+08  2.336924e+08               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
6   ts-admin-basic-info-service      socket           25  6.616667e+00  7.680000e+00               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
7   ts-admin-basic-info-service    workload           25  6.268000e-01  7.186300e-01               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
8        ts-admin-order-service         cpu           25  1.104006e-01  1.262462e-01               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
9        ts-admin-order-service      diskio           25  5.134890e+04  6.237697e+04               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
10       ts-admin-order-service         mem           25  1.929585e+08  1.930392e+08               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
11       ts-admin-order-service      socket           25  3.000000e+00  3.300000e+00               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
12       ts-admin-route-service         cpu           25  1.037472e-01  1.175940e-01               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
13       ts-admin-route-service      diskio           25  3.984585e+04  5.033690e+04               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
14       ts-admin-route-service         mem           25  1.951711e+08  1.952872e+08               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
15       ts-admin-route-service      socket           25  3.000000e+00  3.463333e+00               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
16      ts-admin-travel-service         cpu           25  3.505769e-01  6.092216e-01               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
17      ts-admin-travel-service      diskio           25  5.319322e+04  5.908697e+04               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
18      ts-admin-travel-service       error           25  0.000000e+00  0.000000e+00               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
19      ts-admin-travel-service  latency-50           25  1.750000e-01  1.840357e-01               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z,                cmdb_id                                         trace_name  count_total   p50_value   p95_value  count_in_window    first_ts_in_window     last_ts_in_window
0   ts-travel2-service              trace.to_ts-travel2-service.row_count           25  607.000000  658.400000               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
1                 root  trace.to_ts-admin-basic-info-service.duration_...           25    0.013082    0.014091               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
2                 root  trace.to_ts-admin-basic-info-service.duration_p95           25    0.015338    0.022707               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
3                 root    trace.to_ts-admin-basic-info-service.error_rate           25    0.000000    0.000000               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
4                 root     trace.to_ts-admin-basic-info-service.row_count           25   18.000000   21.000000               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
5                 root     trace.to_ts-admin-travel-service.duration_mean           25    0.152156    0.170248               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
6                 root      trace.to_ts-admin-travel-service.duration_p95           25    0.168879    0.247169               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
7                 root        trace.to_ts-admin-travel-service.error_rate           25    0.000000    0.000000               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
8                 root         trace.to_ts-admin-travel-service.row_count           25    9.000000   11.000000               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
9                 root        trace.to_ts-assurance-service.duration_mean           25    0.003908    0.004728               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
10                root         trace.to_ts-assurance-service.duration_p95           25    0.004482    0.007450               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
11                root           trace.to_ts-assurance-service.error_rate           25    0.000000    0.000000               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
12                root            trace.to_ts-assurance-service.row_count           25    7.000000   11.600000               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
13                root             trace.to_ts-auth-service.duration_mean           25    0.282377    0.398266               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
14                root              trace.to_ts-auth-service.duration_p95           25    0.597169    1.015229               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
15                root                trace.to_ts-auth-service.error_rate           25    0.000000    0.000000               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
16   ts-travel-service       trace.to_ts-ticketinfo-service.duration_mean           25    0.011642    0.012358               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
17   ts-travel-service        trace.to_ts-ticketinfo-service.duration_p95           25    0.015506    0.017132               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
18   ts-travel-service          trace.to_ts-ticketinfo-service.error_rate           25    0.000000    0.000000               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
19   ts-travel-service           trace.to_ts-ticketinfo-service.row_count           25   44.000000   59.200000               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z,                         cmdb_id         log_name  count_total  p50_value  p95_value  count_in_window    first_ts_in_window     last_ts_in_window
0   ts-admin-basic-info-service  log.error_count           25        0.0        0.0               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
1   ts-admin-basic-info-service    log.row_count           25       28.0       32.0               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
2       ts-admin-travel-service  log.error_count           25        0.0        0.0               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
3       ts-admin-travel-service    log.row_count           25       27.0       33.0               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
4          ts-assurance-service    log.row_count           25        7.0       11.6               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
5          ts-assurance-service  log.error_count           25        0.0        0.0               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
6              ts-basic-service  log.error_count           25        0.0        0.0               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
7              ts-basic-service    log.row_count           25      214.0      230.8               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
8               ts-auth-service    log.row_count           25      358.0      379.2               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
9               ts-auth-service  log.error_count           25        0.0        0.0               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
10          ts-food-map-service    log.row_count           25       19.0       23.8               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
11          ts-food-map-service  log.error_count           25        0.0        0.0               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
12           ts-consign-service    log.row_count           25       42.0       75.6               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
13           ts-consign-service  log.error_count           25        0.0        0.0               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
14            ts-travel-service  log.error_count           25        0.0        0.0               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
15            ts-travel-service    log.row_count           25      451.0      558.6               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
16           ts-travel2-service  log.error_count           25        0.0        0.0               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
17           ts-travel2-service    log.row_count           25      504.0      553.0               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
18    ts-inside-payment-service  log.error_count           25        0.0        0.0               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z
19    ts-inside-payment-service    log.row_count           25       12.0       23.2               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z, Empty DataFrame
Columns: [cmdb_id, count_total, count_in_window, first_ts_in_window, last_ts_in_window]
Index: [], 0                             root
1      ts-admin-basic-info-service
2           ts-admin-order-service
3           ts-admin-route-service
4          ts-admin-travel-service
                  ...             
64                   ts-user-mongo
65                 ts-user-service
66    ts-verification-code-service
67                ts-voucher-mysql
68              ts-voucher-service
Name: cmdb_id, Length: 69, dtype: object)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# reuse existing df_metric, window_start, window_end if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Ensure window_start/window_end exist
try:
    window_start, window_end
except NameError:
    window_start = pd.to_datetime("2024-01-24 04:08:41", utc=True)
    window_end = pd.to_datetime("2024-01-24 04:38:41", utc=True)

# 1) Compute global p95 and p5 per (cmdb_id, kpi_name) using the full series
thresholds = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(p95_value=lambda x: x.quantile(0.95), p5_value=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Filter original metric points to the incident window
df_metric_win = df_metric[(df_metric['ts'] >= window_start) & (df_metric['ts'] <= window_end)].copy()

# 3) Window-level stats (max/min across the whole window per group)
window_stats = (
    df_metric_win
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(count_in_window='size', max_value_in_window='max', min_value_in_window='min')
    .reset_index()
)

# Merge thresholds with window stats to have p95/p5 and window min/max together
group_df = pd.merge(thresholds, window_stats, on=['cmdb_id', 'kpi_name'], how='left')

# Attach thresholds to each row in window to find anomaly rows
merged_win = pd.merge(df_metric_win, thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# High anomalies: value >= p95
high_rows = merged_win[merged_win['value'] >= merged_win['p95_value']].copy()
high_agg = (
    high_rows
    .groupby(['cmdb_id', 'kpi_name'])
    .agg(count_anomalies_in_window=('value', 'size'),
         first_anomaly_timestamp_in_window=('ts', 'min'),
         last_anomaly_timestamp_in_window=('ts', 'max'))
    .reset_index()
)

# Low anomalies: value <= p5
low_rows = merged_win[merged_win['value'] <= merged_win['p5_value']].copy()
low_agg = (
    low_rows
    .groupby(['cmdb_id', 'kpi_name'])
    .agg(count_anomalies_in_window=('value', 'size'),
         first_anomaly_timestamp_in_window=('ts', 'min'),
         last_anomaly_timestamp_in_window=('ts', 'max'))
    .reset_index()
)

# Prepare high anomaly summary by merging with group_df to get p95/p5 and window min/max
high_summary = pd.merge(high_agg, group_df, on=['cmdb_id', 'kpi_name'], how='left')
# Only keep rows where count_anomalies_in_window > 0 (grouped ones already satisfy)
high_summary['anomaly_type'] = 'high'
# Severity = (max_value_in_window - p95_value) / p95_value
high_summary['severity_metric'] = (high_summary['max_value_in_window'] - high_summary['p95_value']) / high_summary['p95_value']
# Handle division by zero or NaN
high_summary['severity_metric'] = high_summary['severity_metric'].replace([np.inf, -np.inf], np.nan)

# Prepare low anomaly summary
low_summary = pd.merge(low_agg, group_df, on=['cmdb_id', 'kpi_name'], how='left')
low_summary['anomaly_type'] = 'low'
# Severity = (p5_value - min_value_in_window) / p5_value
low_summary['severity_metric'] = (low_summary['p5_value'] - low_summary['min_value_in_window']) / low_summary['p5_value']
low_summary['severity_metric'] = low_summary['severity_metric'].replace([np.inf, -np.inf], np.nan)

# Select/rename columns to the requested schema
select_cols = [
    'cmdb_id', 'kpi_name', 'p95_value', 'p5_value', 'anomaly_type',
    'count_anomalies_in_window', 'first_anomaly_timestamp_in_window', 'last_anomaly_timestamp_in_window',
    'max_value_in_window', 'min_value_in_window', 'severity_metric'
]

high_summary = high_summary[select_cols]
low_summary = low_summary[select_cols]

# Format timestamp columns to compact ISO strings
for df in (high_summary, low_summary):
    df['first_anomaly_timestamp_in_window'] = df['first_anomaly_timestamp_in_window'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    df['last_anomaly_timestamp_in_window'] = df['last_anomaly_timestamp_in_window'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Combine high and low summaries
metric_anom_res = pd.concat([high_summary, low_summary], ignore_index=True, sort=False)

# Drop groups with no anomalies (shouldn't be present), and sort by count desc then severity desc, limit top 20
metric_anom_res = metric_anom_res.dropna(subset=['count_anomalies_in_window'])
metric_anom_res['count_anomalies_in_window'] = metric_anom_res['count_anomalies_in_window'].astype(int)
metric_anom_res = metric_anom_res.sort_values(['count_anomalies_in_window', 'severity_metric'], ascending=[False, False]).head(20).reset_index(drop=True)

# Keep final output compact: selected columns and head(20) already applied
metric_anom_res

```
Out[2]:
```
Summary of metric anomaly detection (window 2024-01-24 04:08:41 → 2024-01-24 04:38:41 UTC)

- What was detected:
  - The top results are dominated by socket metrics from many mongo-related services (e.g., ts-assurance-mongo, ts-auth-mongo, ts-order-mongo, ts-route-mongo, etc.).
  - All reported anomalies are labeled "high" (value >= global P95) and each group has count_anomalies_in_window = 25 (samples cover the window).
  - For every reported group, p95_value == p5_value and max_value_in_window == min_value_in_window == that same value.

- Timing / coverage:
  - For the top groups the first anomaly sample is at 2024-01-24T04:11:00Z and the last at 2024-01-24T04:35:00Z, i.e., the anomalies span the available samples inside the incident window.

- Severity:
  - severity_metric is 0.0 for all reported rows because the window max equals the P95 (for high anomalies) — there is no measured exceedance above the P95.
  - In short, these metrics are constant at the threshold value (no spike or drop beyond the computed global thresholds).

- Interpretation:
  - The groups flagged are not showing true outlier spikes or drops during the window — they are stable and equal to their global P5/P95. The anomaly counts reflect that values meet the threshold condition, not that they exceed it.
  - No high-severity deviations were found in the top 20 groups returned. If further investigation is needed, focus on metrics with nonzero severity_metric or groups where p95 and p5 differ (indicating variability).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id kpi_name    p95_value     p5_value anomaly_type  count_anomalies_in_window first_anomaly_timestamp_in_window last_anomaly_timestamp_in_window  max_value_in_window  min_value_in_window  severity_metric
0        ts-assurance-mongo   socket          3.0          3.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  3.0                  3.0              0.0
1             ts-auth-mongo   socket          6.0          6.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  6.0                  6.0              0.0
2         ts-avatar-service   socket          5.0          5.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  5.0                  5.0              0.0
3           ts-config-mongo   socket          4.0          4.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  4.0                  4.0              0.0
4    ts-consign-price-mongo   socket          4.0          4.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  4.0                  4.0              0.0
5         ts-contacts-mongo   socket          3.0          3.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  3.0                  3.0              0.0
6         ts-food-map-mongo   socket          5.0          5.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  5.0                  5.0              0.0
7             ts-food-mongo   socket          3.0          3.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  3.0                  3.0              0.0
8             ts-food-mysql      mem  599805952.0  599805952.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z          599805952.0          599805952.0              0.0
9             ts-food-mysql   socket          2.0          2.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  2.0                  2.0              0.0
10  ts-inside-payment-mongo   socket          3.0          3.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  3.0                  3.0              0.0
11          ts-news-service   socket          1.0          1.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  1.0                  1.0              0.0
12    ts-notification-mongo   socket          2.0          2.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  2.0                  2.0              0.0
13           ts-order-mongo   socket          5.0          5.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  5.0                  5.0              0.0
14     ts-order-other-mongo   socket          4.0          4.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  4.0                  4.0              0.0
15         ts-payment-mongo   socket          3.0          3.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  3.0                  3.0              0.0
16        ts-preserve-mongo   socket          2.0          2.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  2.0                  2.0              0.0
17  ts-preserve-other-mongo   socket          2.0          2.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  2.0                  2.0              0.0
18           ts-price-mongo   socket          4.0          4.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  4.0                  4.0              0.0
19           ts-route-mongo   socket          5.0          5.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  5.0                  5.0              0.0```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse loaded DataFrames if present; otherwise load files
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")
try:
    df_err
except NameError:
    df_err = pd.read_csv("error_logs.csv")

# Ensure timestamp -> UTC datetime column 'ts' exists
for df in (df_metric, df_trace, df_log, df_err):
    if 'ts' not in df.columns:
        df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Incident window (UTC)
window_start = pd.to_datetime("2024-01-24 04:08:41", utc=True)
window_end = pd.to_datetime("2024-01-24 04:38:41", utc=True)

# Candidate services
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Helper: format timestamps safely
def fmt_ts_col(s):
    if s.dtype == object:
        s = pd.to_datetime(s, utc=True, errors='coerce')
    return s.dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# -----------------------
# 1) Metrics (candidates)
# -----------------------
df_metric_cand = df_metric[df_metric['cmdb_id'].isin(candidates)].copy()

# Global thresholds (full series for these groups)
metric_thresholds = (
    df_metric_cand
    .groupby(['cmdb_id','kpi_name'])['value']
    .agg(p95_value=lambda x: x.quantile(0.95), p5_value=lambda x: x.quantile(0.05))
    .reset_index()
)

# Windowed metric points
df_metric_win = df_metric_cand[(df_metric_cand['ts'] >= window_start) & (df_metric_cand['ts'] <= window_end)].copy()

# Window stats per group (needed for severity)
metric_window_stats = (
    df_metric_win
    .groupby(['cmdb_id','kpi_name'])['value']
    .agg(count_in_window='size', max_value_in_window='max', min_value_in_window='min')
    .reset_index()
)

# Merge thresholds with window stats
metric_groups = pd.merge(metric_thresholds, metric_window_stats, on=['cmdb_id','kpi_name'], how='left')

# Merge thresholds into window rows for detection
metric_win_merged = pd.merge(df_metric_win, metric_thresholds, on=['cmdb_id','kpi_name'], how='left')

# Detect high and low anomalies
metric_high = metric_win_merged[metric_win_merged['value'] >= metric_win_merged['p95_value']].copy()
metric_low  = metric_win_merged[metric_win_merged['value'] <= metric_win_merged['p5_value']].copy()

# Aggregate anomalies
metric_high_agg = (
    metric_high
    .groupby(['cmdb_id','kpi_name'])
    .agg(count_anomalies_in_window=('value','size'),
         first_anomaly_timestamp_in_window=('ts','min'),
         last_anomaly_timestamp_in_window=('ts','max'))
    .reset_index()
)
metric_low_agg = (
    metric_low
    .groupby(['cmdb_id','kpi_name'])
    .agg(count_anomalies_in_window=('value','size'),
         first_anomaly_timestamp_in_window=('ts','min'),
         last_anomaly_timestamp_in_window=('ts','max'))
    .reset_index()
)

metric_high_summary = pd.merge(metric_high_agg, metric_groups, on=['cmdb_id','kpi_name'], how='left')
metric_low_summary  = pd.merge(metric_low_agg,  metric_groups, on=['cmdb_id','kpi_name'], how='left')

# Add anomaly_type and severity
metric_high_summary['anomaly_type'] = 'high'
metric_high_summary['severity_metric'] = (metric_high_summary['max_value_in_window'] - metric_high_summary['p95_value']) / metric_high_summary['p95_value']
metric_low_summary['anomaly_type'] = 'low'
metric_low_summary['severity_metric'] = (metric_low_summary['p5_value'] - metric_low_summary['min_value_in_window']) / metric_low_summary['p5_value']

# Clean up infinities/nans
metric_high_summary['severity_metric'] = metric_high_summary['severity_metric'].replace([np.inf, -np.inf], np.nan)
metric_low_summary['severity_metric'] = metric_low_summary['severity_metric'].replace([np.inf, -np.inf], np.nan)

# Format timestamps
metric_high_summary['first_anomaly_timestamp_in_window'] = fmt_ts_col(metric_high_summary['first_anomaly_timestamp_in_window'])
metric_high_summary['last_anomaly_timestamp_in_window']  = fmt_ts_col(metric_high_summary['last_anomaly_timestamp_in_window'])
metric_low_summary['first_anomaly_timestamp_in_window']  = fmt_ts_col(metric_low_summary['first_anomaly_timestamp_in_window'])
metric_low_summary['last_anomaly_timestamp_in_window']   = fmt_ts_col(metric_low_summary['last_anomaly_timestamp_in_window'])

metric_res = pd.concat([metric_high_summary, metric_low_summary], ignore_index=True, sort=False)
metric_res = metric_res[[
    'cmdb_id','kpi_name','p95_value','p5_value','anomaly_type',
    'count_anomalies_in_window','first_anomaly_timestamp_in_window','last_anomaly_timestamp_in_window',
    'max_value_in_window','min_value_in_window','severity_metric'
]]
metric_res = metric_res.dropna(subset=['count_anomalies_in_window'])
metric_res['count_anomalies_in_window'] = metric_res['count_anomalies_in_window'].astype(int)
metric_res = metric_res.sort_values(['count_anomalies_in_window','severity_metric'], ascending=[False, False]).head(20).reset_index(drop=True)

# -----------------------
# 2) Traces (candidates)
# -----------------------
def trace_name_matches(name):
    if pd.isna(name):
        return False
    for s in candidates:
        if s in name:
            return True
    return False

mask_trace = df_trace['cmdb_id'].isin(candidates) | df_trace['trace_name'].apply(trace_name_matches)
df_trace_cand = df_trace[mask_trace].copy()

trace_thresholds = (
    df_trace_cand
    .groupby(['cmdb_id','trace_name'])['value']
    .agg(p95_value=lambda x: x.quantile(0.95), p5_value=lambda x: x.quantile(0.05))
    .reset_index()
)

df_trace_win = df_trace_cand[(df_trace_cand['ts'] >= window_start) & (df_trace_cand['ts'] <= window_end)].copy()

trace_window_stats = (
    df_trace_win
    .groupby(['cmdb_id','trace_name'])['value']
    .agg(count_in_window='size', max_value_in_window='max', min_value_in_window='min')
    .reset_index()
)

trace_groups = pd.merge(trace_thresholds, trace_window_stats, on=['cmdb_id','trace_name'], how='left')
trace_win_merged = pd.merge(df_trace_win, trace_thresholds, on=['cmdb_id','trace_name'], how='left')

trace_high = trace_win_merged[trace_win_merged['value'] >= trace_win_merged['p95_value']].copy()
trace_low  = trace_win_merged[trace_win_merged['value'] <= trace_win_merged['p5_value']].copy()

trace_high_agg = (
    trace_high
    .groupby(['cmdb_id','trace_name'])
    .agg(count_anomalies_in_window=('value','size'),
         first_anomaly_timestamp_in_window=('ts','min'),
         last_anomaly_timestamp_in_window=('ts','max'))
    .reset_index()
)
trace_low_agg = (
    trace_low
    .groupby(['cmdb_id','trace_name'])
    .agg(count_anomalies_in_window=('value','size'),
         first_anomaly_timestamp_in_window=('ts','min'),
         last_anomaly_timestamp_in_window=('ts','max'))
    .reset_index()
)

trace_high_summary = pd.merge(trace_high_agg, trace_groups, on=['cmdb_id','trace_name'], how='left')
trace_low_summary  = pd.merge(trace_low_agg,  trace_groups, on=['cmdb_id','trace_name'], how='left')

trace_high_summary['anomaly_type'] = 'high'
trace_high_summary['severity_metric'] = (trace_high_summary['max_value_in_window'] - trace_high_summary['p95_value']) / trace_high_summary['p95_value']
trace_low_summary['anomaly_type'] = 'low'
trace_low_summary['severity_metric'] = (trace_low_summary['p5_value'] - trace_low_summary['min_value_in_window']) / trace_low_summary['p5_value']

trace_high_summary['severity_metric'] = trace_high_summary['severity_metric'].replace([np.inf, -np.inf], np.nan)
trace_low_summary['severity_metric']  = trace_low_summary['severity_metric'].replace([np.inf, -np.inf], np.nan)

trace_high_summary['first_anomaly_timestamp_in_window'] = fmt_ts_col(trace_high_summary['first_anomaly_timestamp_in_window'])
trace_high_summary['last_anomaly_timestamp_in_window']  = fmt_ts_col(trace_high_summary['last_anomaly_timestamp_in_window'])
trace_low_summary['first_anomaly_timestamp_in_window']  = fmt_ts_col(trace_low_summary['first_anomaly_timestamp_in_window'])
trace_low_summary['last_anomaly_timestamp_in_window']   = fmt_ts_col(trace_low_summary['last_anomaly_timestamp_in_window'])

trace_res = pd.concat([trace_high_summary, trace_low_summary], ignore_index=True, sort=False)
trace_res = trace_res[[
    'cmdb_id','trace_name','p95_value','p5_value','anomaly_type',
    'count_anomalies_in_window','first_anomaly_timestamp_in_window','last_anomaly_timestamp_in_window',
    'max_value_in_window','min_value_in_window','severity_metric'
]]
trace_res = trace_res.dropna(subset=['count_anomalies_in_window'])
trace_res['count_anomalies_in_window'] = trace_res['count_anomalies_in_window'].astype(int)
trace_res = trace_res.sort_values(['count_anomalies_in_window','severity_metric'], ascending=[False, False]).head(20).reset_index(drop=True)

# -----------------------
# 3) Logs (candidates)
# -----------------------
df_log_cand = df_log[df_log['cmdb_id'].isin(candidates)].copy()

log_thresholds = (
    df_log_cand
    .groupby(['cmdb_id','log_name'])['value']
    .agg(p95_value=lambda x: x.quantile(0.95), p5_value=lambda x: x.quantile(0.05))
    .reset_index()
)

df_log_win = df_log_cand[(df_log_cand['ts'] >= window_start) & (df_log_cand['ts'] <= window_end)].copy()

# Compute window stats for logs as well (for severity)
log_window_stats = (
    df_log_win
    .groupby(['cmdb_id','log_name'])['value']
    .agg(count_in_window='size', max_value_in_window='max', min_value_in_window='min')
    .reset_index()
)

log_groups = pd.merge(log_thresholds, log_window_stats, on=['cmdb_id','log_name'], how='left')
log_win_merged = pd.merge(df_log_win, log_thresholds, on=['cmdb_id','log_name'], how='left')

# Decide anomaly direction per log_name
is_error = log_win_merged['log_name'].str.contains('error', case=False, na=False)
is_row   = log_win_merged['log_name'].str.contains('row', case=False, na=False)

log_high = log_win_merged[is_error & (log_win_merged['value'] >= log_win_merged['p95_value'])].copy()
log_low  = log_win_merged[is_row &   (log_win_merged['value'] <= log_win_merged['p5_value'])].copy()

log_high_agg = (
    log_high
    .groupby(['cmdb_id','log_name'])
    .agg(count_anomalies_in_window=('value','size'),
         first_anomaly_timestamp_in_window=('ts','min'),
         last_anomaly_timestamp_in_window=('ts','max'))
    .reset_index()
)
log_low_agg = (
    log_low
    .groupby(['cmdb_id','log_name'])
    .agg(count_anomalies_in_window=('value','size'),
         first_anomaly_timestamp_in_window=('ts','min'),
         last_anomaly_timestamp_in_window=('ts','max'))
    .reset_index()
)

log_high_summary = pd.merge(log_high_agg, log_groups, on=['cmdb_id','log_name'], how='left')
log_low_summary  = pd.merge(log_low_agg,  log_groups, on=['cmdb_id','log_name'], how='left')

log_high_summary['anomaly_type'] = 'high'
log_high_summary['severity_metric'] = (log_high_summary['max_value_in_window'] - log_high_summary['p95_value']) / log_high_summary['p95_value']
log_low_summary['anomaly_type'] = 'low'
log_low_summary['severity_metric'] = (log_low_summary['p5_value'] - log_low_summary['min_value_in_window']) / log_low_summary['p5_value']

log_high_summary['severity_metric'] = log_high_summary['severity_metric'].replace([np.inf, -np.inf], np.nan)
log_low_summary['severity_metric']  = log_low_summary['severity_metric'].replace([np.inf, -np.inf], np.nan)

log_high_summary['first_anomaly_timestamp_in_window'] = fmt_ts_col(log_high_summary['first_anomaly_timestamp_in_window'])
log_high_summary['last_anomaly_timestamp_in_window']  = fmt_ts_col(log_high_summary['last_anomaly_timestamp_in_window'])
log_low_summary['first_anomaly_timestamp_in_window'] = fmt_ts_col(log_low_summary['first_anomaly_timestamp_in_window'])
log_low_summary['last_anomaly_timestamp_in_window']  = fmt_ts_col(log_low_summary['last_anomaly_timestamp_in_window'])

log_res = pd.concat([log_high_summary, log_low_summary], ignore_index=True, sort=False)
log_res = log_res[[
    'cmdb_id','log_name','p95_value','p5_value','anomaly_type',
    'count_anomalies_in_window','first_anomaly_timestamp_in_window','last_anomaly_timestamp_in_window',
    'max_value_in_window','min_value_in_window','severity_metric'
]]
log_res = log_res.dropna(subset=['count_anomalies_in_window'])
log_res['count_anomalies_in_window'] = log_res['count_anomalies_in_window'].astype(int)
log_res = log_res.sort_values(['count_anomalies_in_window','severity_metric'], ascending=[False, False]).head(20).reset_index(drop=True)

# -----------------------
# 4) Error logs (raw rows)
# -----------------------
err_rows = df_err[(df_err['cmdb_id'].isin(candidates)) & (df_err['ts'] >= window_start) & (df_err['ts'] <= window_end)].copy()
if not err_rows.empty:
    err_rows_view = err_rows[['timestamp','cmdb_id','message']].copy()
    err_rows_view['timestamp'] = pd.to_datetime(err_rows_view['timestamp'], unit='s', utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    err_count = int(err_rows_view.shape[0])
    err_rows_view = err_rows_view.head(20).reset_index(drop=True)
else:
    err_rows_view = err_rows[['timestamp','cmdb_id','message']].head(0)
    err_count = 0

# Return compact aggregated tables and error log rows summary
metric_res, trace_res, log_res, err_rows_view, err_count

```
Out[3]:
```
Summary (candidate services: ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service; window 2024-01-24 04:08:41 → 04:38:41 UTC)

1) Metrics — key findings
- ts-order-service shows the most concerning metric anomalies:
  - latency-50: 2 high-anomaly samples in-window, severity ≈ 3.80 (max ≈ 0.0933 vs p95 ≈ 0.01944) — the largest relative spike.
  - diskio: 2 high samples, severity ≈ 1.28 (max ≈ 138964 vs p95 ≈ 60936).
  - cpu: 2 high samples, severity ≈ 0.98 (max ≈ 1.288 vs p95 ≈ 0.6497).
  - latency-90: 2 high samples, severity ≈ 0.60.
  - Also some low/high workload and latency-50/latency-50 low occurrences (count=2 each).
  - There are also entries for ts-order-service.error with count=25 but p95=p5=0 (no meaningful severity).
- ts-train-service:
  - socket shows many low anomalies (count 18) within the window (first 04:11, last 04:35). Severity is modest (~0.076) — values are below the global P5 but not hugely different.
  - cpu and workload show 2 low anomalies each with small-to-moderate severity (~0.086 and ~0.064).
- ts-auth-service:
  - latency-90 and diskio and cpu and socket have 2 high-anomaly samples each with moderate severity (latency-90 severity ~0.337, diskio ~0.121, cpu ~0.108, socket ~0.093).
- ts-travel-service:
  - diskio, cpu, workload show 2 anomalies each; some are low (diskio, workload) and some high (cpu) with moderate severity (cpu severity ~0.163).
- ts-route-service also appears in a few metric anomaly rows (smaller counts/severity).

Interpretation: ts-order-service exhibits the largest relative spikes (latency and resource metrics) and thus is the most prominent candidate for performance impact in this window.

2) Traces — key findings
- Many trace entries are flagged because trace error_rate metrics are zero and match p95/p5 = 0 (count 25). These produce no meaningful severity (NaN) because values equal the thresholds.
- No trace anomaly with substantive nonzero severity was identified for the five services in the top results — error rates are zero throughout the window.

3) Logs — key findings
- log.error_count for auth/order/route/travel are zero (p95=p5=0) and flagged but have no meaningful severity.
- log.row_count shows short low-count anomalies (count = 2) for several services:
  - ts-auth-service.log.row_count: p95≈379.2, p5≈252.8, min in-window 114 → severity ≈ 0.55 (substantial drop).
  - ts-route-service.log.row_count: severity ≈ 0.45 (drop).
  - ts-travel-service.log.row_count: severity ≈ 0.295 (drop).
  - ts-order-service.log.row_count: small drop, lower severity (~0.05).
Interpretation: auth and route show notable drops in row_count during the window (reduced throughput/requests), which may be related to observed metric anomalies.

4) error_logs.csv
- No raw error_log rows for these five services were found inside the incident window (0 rows).

Overall concise conclusion
- The strongest signal of problematic behavior is in ts-order-service: multiple metric spikes (latency-50, disk I/O, CPU, latency-90) with the largest relative severity (latency-50 severity ≈ 3.8). This points to ts-order-service as the primary suspect for performance impact in the incident window.
- Secondary signals: notable drops in log.row_count for ts-auth-service and ts-route-service (significant severity), and many low socket values for ts-train-service (countable but lower severity).
- Trace error rates are zero and do not indicate trace-level failures, and there are no error_log entries in the window.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name     p95_value      p5_value anomaly_type  count_anomalies_in_window first_anomaly_timestamp_in_window last_anomaly_timestamp_in_window  max_value_in_window  min_value_in_window  severity_metric
0    ts-order-service       error      0.000000      0.000000         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z             0.000000             0.000000              NaN
1    ts-order-service       error      0.000000      0.000000          low                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z             0.000000             0.000000              NaN
2    ts-train-service      socket      9.310000      9.000000          low                         18              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z             9.483333             8.315789         0.076023
3    ts-order-service  latency-50      0.019444      0.012519         high                          2              2024-01-24T04:30:00Z             2024-01-24T04:32:00Z             0.093333             0.010546         3.800205
4    ts-order-service      diskio  60935.961400  43339.776525         high                          2              2024-01-24T04:12:00Z             2024-01-24T04:32:00Z        138964.215360         42943.529746         1.280496
5    ts-order-service         cpu      0.649702      0.182138         high                          2              2024-01-24T04:12:00Z             2024-01-24T04:23:00Z             1.288492             0.168685         0.983203
6    ts-order-service  latency-90      1.676232      0.022597         high                          2              2024-01-24T04:30:00Z             2024-01-24T04:32:00Z             2.678000             0.022511         0.597631
7    ts-order-service    workload      0.615213      0.228453          low                          2              2024-01-24T04:11:00Z             2024-01-24T04:27:00Z             0.723350             0.098158         0.570337
8     ts-auth-service  latency-90      0.905420      0.447729         high                          2              2024-01-24T04:13:00Z             2024-01-24T04:25:00Z             1.210394             0.441883         0.336832
9   ts-travel-service      diskio  57015.212839  44009.877306          low                          2              2024-01-24T04:11:00Z             2024-01-24T04:13:00Z         57930.785517         32499.234975         0.261547
10   ts-order-service    workload      0.615213      0.228453         high                          2              2024-01-24T04:18:00Z             2024-01-24T04:28:00Z             0.723350             0.098158         0.175771
11  ts-travel-service         cpu      1.167231      0.834516         high                          2              2024-01-24T04:12:00Z             2024-01-24T04:27:00Z             1.358033             0.819639         0.163465
12   ts-order-service  latency-50      0.019444      0.012519          low                          2              2024-01-24T04:11:00Z             2024-01-24T04:14:00Z             0.093333             0.010546         0.157562
13    ts-auth-service      diskio  60394.224465  41440.153218         high                          2              2024-01-24T04:30:00Z             2024-01-24T04:33:00Z         67703.631522         40161.472503         0.121028
14    ts-auth-service         cpu     16.126394     14.103216         high                          2              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z            17.873806            13.519124         0.108357
15    ts-auth-service      socket     14.806667     12.036667         high                          2              2024-01-24T04:13:00Z             2024-01-24T04:25:00Z            16.183333            12.000000         0.092976
16   ts-train-service         cpu      1.320850      0.608020          low                          2              2024-01-24T04:26:00Z             2024-01-24T04:35:00Z             1.383991             0.555465         0.086436
17   ts-order-service         cpu      0.649702      0.182138          low                          2              2024-01-24T04:27:00Z             2024-01-24T04:30:00Z             1.288492             0.168685         0.073862
18  ts-travel-service    workload      2.004677      1.553863          low                          2              2024-01-24T04:31:00Z             2024-01-24T04:34:00Z             2.094842             1.445483         0.069748
19   ts-train-service    workload      3.786713      2.666367          low                          2              2024-01-24T04:22:00Z             2024-01-24T04:31:00Z             3.828883             2.494550         0.064438,                     cmdb_id                                trace_name  p95_value  p5_value anomaly_type  count_anomalies_in_window first_anomaly_timestamp_in_window last_anomaly_timestamp_in_window  max_value_in_window  min_value_in_window  severity_metric
0                      root       trace.to_ts-auth-service.error_rate        0.0       0.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  0.0                  0.0              NaN
1                      root      trace.to_ts-order-service.error_rate        0.0       0.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  0.0                  0.0              NaN
2                      root     trace.to_ts-travel-service.error_rate        0.0       0.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  0.0                  0.0              NaN
3   ts-admin-travel-service     trace.to_ts-travel-service.error_rate        0.0       0.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  0.0                  0.0              NaN
4           ts-auth-service                trace.from_root.error_rate        0.0       0.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  0.0                  0.0              NaN
5           ts-auth-service     trace.from_ts-auth-service.error_rate        0.0       0.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  0.0                  0.0              NaN
6           ts-auth-service       trace.to_ts-auth-service.error_rate        0.0       0.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  0.0                  0.0              NaN
7           ts-food-service     trace.to_ts-travel-service.error_rate        0.0       0.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  0.0                  0.0              NaN
8          ts-order-service                trace.from_root.error_rate        0.0       0.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  0.0                  0.0              NaN
9          ts-order-service    trace.from_ts-order-service.error_rate        0.0       0.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  0.0                  0.0              NaN
10         ts-order-service      trace.to_ts-order-service.error_rate        0.0       0.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  0.0                  0.0              NaN
11         ts-order-service    trace.to_ts-station-service.error_rate        0.0       0.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  0.0                  0.0              NaN
12         ts-route-service    trace.from_ts-route-service.error_rate        0.0       0.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  0.0                  0.0              NaN
13         ts-route-service   trace.from_ts-travel-service.error_rate        0.0       0.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  0.0                  0.0              NaN
14         ts-route-service  trace.from_ts-travel2-service.error_rate        0.0       0.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  0.0                  0.0              NaN
15         ts-route-service      trace.to_ts-route-service.error_rate        0.0       0.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  0.0                  0.0              NaN
16       ts-station-service    trace.from_ts-order-service.error_rate        0.0       0.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  0.0                  0.0              NaN
17    ts-ticketinfo-service   trace.from_ts-travel-service.error_rate        0.0       0.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  0.0                  0.0              NaN
18         ts-train-service    trace.from_ts-train-service.error_rate        0.0       0.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  0.0                  0.0              NaN
19         ts-train-service   trace.from_ts-travel-service.error_rate        0.0       0.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  0.0                  0.0              NaN,              cmdb_id         log_name  p95_value  p5_value anomaly_type  count_anomalies_in_window first_anomaly_timestamp_in_window last_anomaly_timestamp_in_window  max_value_in_window  min_value_in_window  severity_metric
0    ts-auth-service  log.error_count        0.0       0.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                    0                    0              NaN
1   ts-order-service  log.error_count        0.0       0.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                    0                    0              NaN
2   ts-route-service  log.error_count        0.0       0.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                    0                    0              NaN
3  ts-travel-service  log.error_count        0.0       0.0         high                         25              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                    0                    0              NaN
4    ts-auth-service    log.row_count      379.2     252.8          low                          2              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  400                  114         0.549051
5   ts-route-service    log.row_count      382.6     235.2          low                          2              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  393                  130         0.447279
6  ts-travel-service    log.row_count      558.6     316.2          low                          2              2024-01-24T04:11:00Z             2024-01-24T04:35:00Z                  647                  223         0.294750
7   ts-order-service    log.row_count       54.8      14.8          low                          2              2024-01-24T04:29:00Z             2024-01-24T04:34:00Z                   76                   14         0.054054, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [], 0)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import re

# Reuse df_trace, window_start, window_end if present
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

if 'ts' not in df_trace.columns:
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Incident window (ensure variables exist)
window_start = pd.to_datetime("2024-01-24 04:08:41", utc=True)
window_end = pd.to_datetime("2024-01-24 04:38:41", utc=True)

# Candidate services and the exact trace_name substrings to match
patterns = [
    'to_ts-order-service.duration_mean',
    'to_ts-order-service.duration_p95',
    'to_ts-auth-service.duration_mean',
    'to_ts-auth-service.duration_p95',
    'to_ts-route-service.duration_mean',
    'to_ts-route-service.duration_p95',
    'to_ts-train-service.duration_mean',
    'to_ts-train-service.duration_p95',
    'to_ts-travel-service.duration_mean',
    'to_ts-travel-service.duration_p95'
]

# Build a regex that matches any of the substrings
regex = '|'.join(re.escape(p) for p in patterns)

# Filter trace rows to the incident window
df_trace_win = df_trace[(df_trace['ts'] >= window_start) & (df_trace['ts'] <= window_end)].copy()

# Select only trace_name entries matching the duration patterns
mask = df_trace_win['trace_name'].astype(str).str.contains(regex, regex=True)
trace_rows = df_trace_win[mask].copy()

# Prepare the limited raw rows (up to 200) sorted by timestamp
if not trace_rows.empty:
    trace_rows = trace_rows.sort_values('ts').head(200).reset_index(drop=True)
    trace_rows_view = trace_rows[['ts','cmdb_id','trace_name','value']].copy()
    trace_rows_view['timestamp'] = trace_rows_view['ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    trace_rows_view = trace_rows_view[['timestamp','cmdb_id','trace_name','value']]
else:
    trace_rows_view = trace_rows[['timestamp','cmdb_id','trace_name','value']].head(0)

# Grouped per-trace_name summary (only for trace_names found)
if not trace_rows.empty:
    trace_summary = (
        trace_rows
        .groupby('trace_name')
        .agg(
            count_in_window=('value','size'),
            first_ts_in_window=('ts','min'),
            last_ts_in_window=('ts','max'),
            min_value_in_window=('value','min'),
            max_value_in_window=('value','max'),
            median_value_in_window=('value','median')
        )
        .reset_index()
    )
    # Format timestamps compactly
    trace_summary['first_ts_in_window'] = trace_summary['first_ts_in_window'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    trace_summary['last_ts_in_window']  = trace_summary['last_ts_in_window'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    # Reorder columns as requested and limit to up to 20 trace_names
    trace_summary = trace_summary[[
        'trace_name','count_in_window','first_ts_in_window','last_ts_in_window',
        'min_value_in_window','max_value_in_window','median_value_in_window'
    ]].head(20).reset_index(drop=True)
else:
    trace_summary = trace_rows[['trace_name']].head(0)

# Return the two compact tables
trace_rows_view, trace_summary

```
Out[4]:
```
Results (trace durations for candidate services in window 2024-01-24 04:08:41 → 04:38:41 UTC)

Raw rows returned
- Up to 200 trace rows were returned, sorted by timestamp (first shown row at 2024-01-24T04:11:00Z, many rows through 2024-01-24T04:17:00Z in the sample).

Per-trace_name summary (only names found in the filtered rows)
- trace.to_ts-auth-service.duration_mean
  - count_in_window: 13, first: 2024-01-24T04:11:00Z, last: 2024-01-24T04:17:00Z
  - min: 0.001801 s, max: 0.468647 s, median: 0.258543 s
- trace.to_ts-auth-service.duration_p95
  - count_in_window: 13, first: 2024-01-24T04:11:00Z, last: 2024-01-24T04:17:00Z
  - min: 0.001810 s, max: 1.700074 s, median: 0.512460 s
  - note: this trace shows the largest single-duration value (~1.7 s) and a high median (~0.51 s)
- trace.to_ts-order-service.duration_mean
  - count_in_window: 19, first: 2024-01-24T04:11:00Z, last: 2024-01-24T04:17:00Z
  - min: 0.003911 s, max: 0.020171 s, median: 0.005703 s
- trace.to_ts-order-service.duration_p95
  - count_in_window: 20, first: 2024-01-24T04:11:00Z, last: 2024-01-24T04:17:00Z
  - min: 0.004793 s, max: 0.026546 s, median: 0.012141 s
- trace.to_ts-route-service.duration_mean
  - count_in_window: 21, first: 2024-01-24T04:11:00Z, last: 2024-01-24T04:17:00Z
  - min: 0.001254 s, max: 0.003132 s, median: 0.002977 s
- trace.to_ts-route-service.duration_p95
  - count_in_window: 21, first: 2024-01-24T04:11:00Z, last: 2024-01-24T04:17:00Z
  - min: 0.001652 s, max: 0.003989 s, median: 0.003594 s
- trace.to_ts-train-service.duration_mean
  - count_in_window: 21, first: 2024-01-24T04:11:00Z, last: 2024-01-24T04:17:00Z
  - min: 0.001158 s, max: 0.009223 s, median: 0.004733 s
- trace.to_ts-train-service.duration_p95
  - count_in_window: 21, first: 2024-01-24T04:11:00Z, last: 2024-01-24T04:17:00Z
  - min: 0.001583 s, max: 0.024466 s, median: 0.005628 s
- trace.to_ts-travel-service.duration_mean
  - count_in_window: 25, first: 2024-01-24T04:11:00Z, last: 2024-01-24T04:17:00Z
  - min: 0.005492 s, max: 0.077704 s, median: 0.011587 s
- trace.to_ts-travel-service.duration_p95
  - count_in_window: 26, first: 2024-01-24T04:11:00Z, last: 2024-01-24T04:17:00Z
  - min: 0.011462 s, max: 0.095228 s, median: 0.039711 s

Concise interpretation
- Auth-target traces (especially trace.to_ts-auth-service.duration_p95) show notably higher durations and variability (max ~1.7 s, median ~0.51 s).
- Travel-target traces also show higher durations than order/route/train (travel p95 median ~0.04 s, max ~0.095 s; mean max ~0.078 s).
- Order, route, and train call durations are relatively low and stable (maxes generally <0.03 s).
- All returned traces in this sample cover timestamps between ~04:11 and ~04:17 UTC within the requested window.

If you want the full (untruncated) set beyond the 200-row sample or additional statistics (e.g., per-cmdb_id breakdown), I can return that.

The original code execution output of IPython Kernel is also provided below for reference:

(                timestamp                    cmdb_id                               trace_name     value
0    2024-01-24T04:11:00Z                       root   trace.to_ts-auth-service.duration_mean  0.371872
1    2024-01-24T04:11:00Z         ts-travel2-service   trace.to_ts-train-service.duration_p95  0.024466
2    2024-01-24T04:11:00Z         ts-travel2-service  trace.to_ts-train-service.duration_mean  0.009223
3    2024-01-24T04:11:00Z         ts-travel2-service   trace.to_ts-route-service.duration_p95  0.003594
4    2024-01-24T04:11:00Z         ts-travel2-service  trace.to_ts-route-service.duration_mean  0.003045
..                    ...                        ...                                      ...       ...
195  2024-01-24T04:17:00Z    ts-admin-travel-service  trace.to_ts-travel-service.duration_p95  0.089815
196  2024-01-24T04:17:00Z  ts-inside-payment-service   trace.to_ts-order-service.duration_p95  0.006011
197  2024-01-24T04:17:00Z                       root   trace.to_ts-auth-service.duration_mean  0.258543
198  2024-01-24T04:17:00Z                       root    trace.to_ts-auth-service.duration_p95  0.520949
199  2024-01-24T04:17:00Z                       root  trace.to_ts-order-service.duration_mean  0.019463

[200 rows x 4 columns],                                  trace_name  count_in_window    first_ts_in_window     last_ts_in_window  min_value_in_window  max_value_in_window  median_value_in_window
0    trace.to_ts-auth-service.duration_mean               13  2024-01-24T04:11:00Z  2024-01-24T04:17:00Z             0.001801             0.468647                0.258543
1     trace.to_ts-auth-service.duration_p95               13  2024-01-24T04:11:00Z  2024-01-24T04:17:00Z             0.001810             1.700074                0.512460
2   trace.to_ts-order-service.duration_mean               19  2024-01-24T04:11:00Z  2024-01-24T04:17:00Z             0.003911             0.020171                0.005703
3    trace.to_ts-order-service.duration_p95               20  2024-01-24T04:11:00Z  2024-01-24T04:17:00Z             0.004793             0.026546                0.012141
4   trace.to_ts-route-service.duration_mean               21  2024-01-24T04:11:00Z  2024-01-24T04:17:00Z             0.001254             0.003132                0.002977
5    trace.to_ts-route-service.duration_p95               21  2024-01-24T04:11:00Z  2024-01-24T04:17:00Z             0.001652             0.003989                0.003594
6   trace.to_ts-train-service.duration_mean               21  2024-01-24T04:11:00Z  2024-01-24T04:17:00Z             0.001158             0.009223                0.004733
7    trace.to_ts-train-service.duration_p95               21  2024-01-24T04:11:00Z  2024-01-24T04:17:00Z             0.001583             0.024466                0.005628
8  trace.to_ts-travel-service.duration_mean               25  2024-01-24T04:11:00Z  2024-01-24T04:17:00Z             0.005492             0.077704                0.011587
9   trace.to_ts-travel-service.duration_p95               26  2024-01-24T04:11:00Z  2024-01-24T04:17:00Z             0.011462             0.095228                0.039711)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse existing df_metric and window variables if present; otherwise load/define
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

try:
    window_start, window_end
except NameError:
    window_start = pd.to_datetime("2024-01-24 04:08:41", utc=True)
    window_end = pd.to_datetime("2024-01-24 04:38:41", utc=True)

# Candidate services
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# 1) Compute global thresholds (P95 and P5) on full series for candidate services
df_metric_cand_full = df_metric[df_metric['cmdb_id'].isin(candidates)].copy()
thresholds = (
    df_metric_cand_full
    .groupby(['cmdb_id','kpi_name'])['value']
    .agg(p95_value=lambda x: x.quantile(0.95), p5_value=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Filter metric points to the incident window for candidates
df_metric_win = df_metric_cand_full[(df_metric_cand_full['ts'] >= window_start) & (df_metric_cand_full['ts'] <= window_end)].copy()

# 3) Merge thresholds into windowed rows to detect anomalies
df_win_with_thresh = pd.merge(df_metric_win, thresholds, on=['cmdb_id','kpi_name'], how='left')

# 4) Determine anomaly type per row and compute per-event severity
# high: value >= p95_value  => severity = (value - p95)/p95
# low:  value <= p5_value   => severity = (p5 - value)/p5
def compute_severity(row):
    v = row['value']
    p95 = row['p95_value']
    p5 = row['p5_value']
    # prefer high if both conditions true (happens when p5==p95==v)
    if pd.notna(p95) and v >= p95:
        if p95 == 0:
            return np.nan, 'high'
        return (v - p95) / p95, 'high'
    if pd.notna(p5) and v <= p5:
        if p5 == 0:
            return np.nan, 'low'
        return (p5 - v) / p5, 'low'
    return np.nan, None

sev_results = df_win_with_thresh.apply(compute_severity, axis=1, result_type='expand')
df_win_with_thresh['severity_metric'] = sev_results[0]
df_win_with_thresh['anomaly_type'] = sev_results[1]

# Keep only anomaly rows
anoms = df_win_with_thresh[df_win_with_thresh['anomaly_type'].notna()].copy()
# Ensure ts column is datetime
anoms['ts'] = pd.to_datetime(anoms['ts'], utc=True)

# 5) Prepare per-service aggregates
# Count anomalies per service
count_by_service = anoms.groupby('cmdb_id').size().reindex(candidates, fill_value=0).rename('count_anomalies_in_window_for_service').reset_index()

# Earliest anomaly per service (if any)
anoms_sorted = anoms.sort_values(['cmdb_id','ts'])
earliest = anoms_sorted.groupby('cmdb_id', sort=False).first().reset_index()
# earliest may not include services with zero anomalies; merge later

# Timestamp of max absolute-severity event per service
anoms['abs_sev'] = anoms['severity_metric'].abs()
maxsev_idx = anoms.groupby('cmdb_id')['abs_sev'].idxmax().dropna()
maxsev = anoms.loc[maxsev_idx].reset_index(drop=True)

# Build final one-row-per-service DataFrame
rows = []
for svc in candidates:
    svc_count = int(count_by_service.loc[count_by_service['cmdb_id']==svc, 'count_anomalies_in_window_for_service'].iat[0])
    if svc in earliest['cmdb_id'].values:
        e = earliest[earliest['cmdb_id']==svc].iloc[0]
        earliest_ts = e['ts']
        kpi_earliest = e['kpi_name']
        type_earliest = e['anomaly_type']
        sev_earliest = e['severity_metric']
        earliest_iso = pd.to_datetime(earliest_ts, utc=True).strftime('%Y-%m-%dT%H:%M:%SZ')
    else:
        earliest_iso = pd.NA
        kpi_earliest = pd.NA
        type_earliest = pd.NA
        sev_earliest = np.nan

    if svc in maxsev['cmdb_id'].values:
        m = maxsev[maxsev['cmdb_id']==svc].iloc[0]
        ts_maxsev_iso = pd.to_datetime(m['ts'], utc=True).strftime('%Y-%m-%dT%H:%M:%SZ')
    else:
        ts_maxsev_iso = pd.NA

    rows.append({
        'cmdb_id': svc,
        'earliest_anomaly_timestamp_in_window': earliest_iso,
        'kpi_name_of_earliest_anomaly': kpi_earliest,
        'anomaly_type_of_earliest_anomaly': type_earliest,
        'severity_metric_of_earliest_anomaly': sev_earliest,
        'count_anomalies_in_window_for_service': svc_count,
        'timestamp_of_max_severity_event_for_service': ts_maxsev_iso
    })

service_summary = pd.DataFrame(rows)

# Convert severity column to numeric and ensure ordering by earliest timestamp (NaT last)
# For sorting, parse earliest timestamps, treating pd.NA as NaT
service_summary['earliest_ts_parsed'] = pd.to_datetime(service_summary['earliest_anomaly_timestamp_in_window'], utc=True, errors='coerce')
service_summary = service_summary.sort_values('earliest_ts_parsed', ascending=True, na_position='last').drop(columns=['earliest_ts_parsed']).reset_index(drop=True)

# Display the compact summary for the five services
service_summary

```
Out[5]:
```
Summary of metric anomalies for the five candidate services (window 2024-01-24 04:08:41 → 04:38:41 UTC)

All five services had their earliest detected anomaly at 2024-01-24T04:11:00Z.

Per-service results:
- ts-auth-service
  - Earliest anomaly: 2024-01-24T04:11:00Z
  - KPI of earliest anomaly: cpu (high)
  - Severity of earliest anomaly: ~0.108
  - Total anomalies for service (all KPIs): 28
  - Timestamp of the service's max-severity event: 2024-01-24T04:13:00Z

- ts-order-service
  - Earliest anomaly: 2024-01-24T04:11:00Z
  - KPI of earliest anomaly: error (high)
  - Severity of earliest anomaly: ~0.158
  - Total anomalies for service (all KPIs): 53
  - Timestamp of the service's max-severity event: 2024-01-24T04:30:00Z

- ts-route-service
  - Earliest anomaly: 2024-01-24T04:11:00Z
  - KPI of earliest anomaly: latency-50 (high)
  - Severity of earliest anomaly: ~0.00085
  - Total anomalies for service (all KPIs): 28
  - Timestamp of the service's max-severity event: 2024-01-24T04:27:00Z

- ts-train-service
  - Earliest anomaly: 2024-01-24T04:11:00Z
  - KPI of earliest anomaly: mem (low)
  - Severity of earliest anomaly: ~0.00088
  - Total anomalies for service (all KPIs): 44
  - Timestamp of the service's max-severity event: 2024-01-24T04:35:00Z

- ts-travel-service
  - Earliest anomaly: 2024-01-24T04:11:00Z
  - KPI of earliest anomaly: diskio (low)
  - Severity of earliest anomaly: ~0.00548
  - Total anomalies for service (all KPIs): 28
  - Timestamp of the service's max-severity event: 2024-01-24T04:13:00Z

Concise takeaway
- ts-order-service shows the largest total anomaly count (53) and its largest-severity event occurs later in the window (04:30), making it the primary suspect for notable issues during this incident window.
- Other services also had many anomaly events (ts-train-service: 44; ts-auth/route/travel: 28 each), but their earliest anomalies all start at 04:11 and their peak-severity timestamps vary.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id earliest_anomaly_timestamp_in_window kpi_name_of_earliest_anomaly anomaly_type_of_earliest_anomaly  severity_metric_of_earliest_anomaly  count_anomalies_in_window_for_service timestamp_of_max_severity_event_for_service
0    ts-auth-service                 2024-01-24T04:11:00Z                          cpu                             high                             0.108357                                     28                        2024-01-24T04:13:00Z
1   ts-order-service                 2024-01-24T04:11:00Z                        error                             high                             0.157562                                     53                        2024-01-24T04:30:00Z
2   ts-route-service                 2024-01-24T04:11:00Z                   latency-50                             high                             0.000845                                     28                        2024-01-24T04:27:00Z
3   ts-train-service                 2024-01-24T04:11:00Z                          mem                              low                             0.000876                                     44                        2024-01-24T04:35:00Z
4  ts-travel-service                 2024-01-24T04:11:00Z                       diskio                              low                             0.005483                                     28                        2024-01-24T04:13:00Z```
```

In [ ]:
# In[6]:

import re
import pandas as pd
import numpy as np

# Reuse df_trace, window_start/window_end, candidates if present
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
if 'ts' not in df_trace.columns:
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

try:
    window_start, window_end
except NameError:
    window_start = pd.to_datetime("2024-01-24 04:08:41", utc=True)
    window_end = pd.to_datetime("2024-01-24 04:38:41", utc=True)

candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# 1) Filter to incident window
df_tr_win = df_trace[(df_trace['ts'] >= window_start) & (df_trace['ts'] <= window_end)].copy()

# 2) Keep rows where trace_name contains 'to_ts-' or 'from_ts-'
mask_dir = df_tr_win['trace_name'].astype(str).str.contains('to_ts-|from_ts-', regex=True, na=False)
df_dir = df_tr_win[mask_dir].copy()

# 3) Parse target_service (after 'to_' or 'from_') and trace_metric_type (suffix after target)
def parse_target_and_metric(name):
    if pd.isna(name):
        return pd.NA, pd.NA
    m = re.search(r'(?:to|from)_(ts-[^\.]+)(?:\.(.*))?', name)
    if not m:
        return pd.NA, pd.NA
    target = m.group(1)
    metric = m.group(2) if m.group(2) is not None else pd.NA
    return target, metric

parsed = df_dir['trace_name'].apply(lambda x: pd.Series(parse_target_and_metric(x), index=['target_service','trace_metric_type']))
df_dir = pd.concat([df_dir.reset_index(drop=True), parsed.reset_index(drop=True)], axis=1)

# 4) Select rows where source cmdb_id is in candidates OR parsed target_service is in candidates
mask_select = df_dir['cmdb_id'].isin(candidates) | df_dir['target_service'].isin(candidates)
df_sel = df_dir[mask_select].copy()

# 5) Prepare extracted columns: timestamp ISO UTC, source_cmdb_id, trace_name, target_service, value, trace_metric_type
df_sel['timestamp'] = df_sel['ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
df_sel['source_cmdb_id'] = df_sel['cmdb_id']

# Normalize trace_metric_type to string (if NaN, set to original suffix extraction using split as fallback)
df_sel['trace_metric_type'] = df_sel['trace_metric_type'].where(df_sel['trace_metric_type'].notna(), 
                                                               df_sel['trace_name'].astype(str).apply(lambda s: s.split('.')[-1] if '.' in s else pd.NA))

# 6) Aggregation by (source_cmdb_id, target_service, trace_metric_type)
agg = (
    df_sel
    .groupby(['source_cmdb_id','target_service','trace_metric_type'], dropna=False)['value']
    .agg(
        count_in_window='size',
        first_ts_in_window=lambda x: (df_sel.loc[x.index, 'ts'].min()),
        last_ts_in_window=lambda x: (df_sel.loc[x.index, 'ts'].max()),
        median_value_in_window='median'
    )
    .reset_index()
)

# Format timestamps in aggregation
agg['first_ts_in_window'] = pd.to_datetime(agg['first_ts_in_window'], utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
agg['last_ts_in_window']  = pd.to_datetime(agg['last_ts_in_window'], utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Reorder columns as requested and sort top 50 by count desc then median desc
agg = agg[[
    'source_cmdb_id','target_service','trace_metric_type',
    'count_in_window','first_ts_in_window','last_ts_in_window','median_value_in_window'
]]
agg = agg.sort_values(['count_in_window','median_value_in_window'], ascending=[False, False]).head(50).reset_index(drop=True)

# 7) Compact lists of unique source_cmdb_id and unique target_service values observed
unique_sources = pd.Series(sorted(agg['source_cmdb_id'].dropna().unique()), name='source_cmdb_id')
unique_targets = pd.Series(sorted([t for t in agg['target_service'].dropna().unique()]), name='target_service')

# Return aggregated table and unique lists
agg, unique_sources, unique_targets

```
Out[6]:
```
Summary of trace-call aggregation (window 2024-01-24 04:08:41 → 04:38:41 UTC)

What I returned
1) Aggregated table (top 50 rows by count, shown columns):
- Each row groups calls by source_cmdb_id → target_service and trace_metric_type (e.g., row_count, duration_mean, duration_p95, error_rate).
- Top groups by count:
  - Self-calls (source == target) with trace_metric_type = row_count have the largest counts (50 samples each, from 2024-01-24T04:11:00Z → 2024-01-24T04:35:00Z).
    - ts-route-service → ts-route-service: median row_count = 674
    - ts-train-service → ts-train-service: median row_count = 360
    - ts-auth-service → ts-auth-service: median row_count = 358
    - ts-travel-service → ts-travel-service: median row_count = 305
    - ts-order-service → ts-order-service: median row_count = 30
  - Many root → target_service and other-service → target_service duration and p95 groups follow, typically with 25 samples and small durations:
    - root → ts-order-service: duration_mean median ≈ 0.02017 s, duration_p95 median ≈ 0.02655 s
    - root → ts-travel-service: duration_mean/p95 medians ≈ 0.04885 s / 0.06487 s
    - root → ts-train-service and root → ts-route-service duration medians are smaller (generally < 0.01–0.03 s).
  - Example other entry: ts-ticketinfo-service → ts-travel-service duration_p95 median ≈ 0.01551 s.

2) Unique sources and targets observed (compact lists)
- Unique source_cmdb_id values (observed in these aggregated rows): root, ts-admin-travel-service, ts-auth-service, ts-food-service, ts-order-service, ts-route-service, ts-station-service, ts-ticketinfo-service, ts-train-service, ts-travel-service, ts-travel2-service.
- Unique target_service values observed: ts-admin-travel-service, ts-auth-service, ts-food-service, ts-order-service, ts-route-service, ts-station-service, ts-ticketinfo-service, ts-train-service, ts-travel-service, ts-travel2-service.

Concise interpretation
- The heaviest trace activity (by row_count) is internal/self traffic for route/train/auth/travel services — especially ts-route-service (median row_count 674) and ts-auth-service (median 358).
- Call durations toward order/route/train are generally small and consistent (< ~0.03 s median), while calls toward travel and auth show larger medians/p95s (travel medians ~0.05–0.06 s; auth p95s were higher in earlier summaries).
- These aggregated groups cover samples across the incident window (earliest 04:11 → latest 04:35 UTC in the top groups).

If you want, I can show the specific top rows again or focus on a particular source→target pair for more detail.

The original code execution output of IPython Kernel is also provided below for reference:

(           source_cmdb_id     target_service trace_metric_type  count_in_window    first_ts_in_window     last_ts_in_window  median_value_in_window
0        ts-route-service   ts-route-service         row_count               50  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z              674.000000
1        ts-train-service   ts-train-service         row_count               50  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z              360.000000
2         ts-auth-service    ts-auth-service         row_count               50  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z              358.000000
3       ts-travel-service  ts-travel-service         row_count               50  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z              305.000000
4        ts-order-service   ts-order-service         row_count               50  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z               30.000000
..                    ...                ...               ...              ...                   ...                   ...                     ...
45                   root  ts-travel-service      duration_p95               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z                0.064866
46                   root  ts-travel-service     duration_mean               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z                0.048850
47                   root   ts-order-service      duration_p95               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z                0.026546
48                   root   ts-order-service     duration_mean               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z                0.020171
49  ts-ticketinfo-service  ts-travel-service      duration_p95               25  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z                0.015506

[50 rows x 7 columns], 0                        root
1     ts-admin-travel-service
2             ts-auth-service
3             ts-food-service
4            ts-order-service
5            ts-route-service
6          ts-station-service
7       ts-ticketinfo-service
8            ts-train-service
9           ts-travel-service
10         ts-travel2-service
Name: source_cmdb_id, dtype: object, 0    ts-admin-travel-service
1            ts-auth-service
2            ts-food-service
3           ts-order-service
4           ts-route-service
5         ts-station-service
6      ts-ticketinfo-service
7           ts-train-service
8          ts-travel-service
9         ts-travel2-service
Name: target_service, dtype: object)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import re
import pandas as pd
import numpy as np

# Reuse variables if present; otherwise reconstruct necessary trace selection
try:
    df_sel
except NameError:
    # Load trace and prepare ts
    df_trace = pd.read_csv("trace.csv")
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
    window_start = pd.to_datetime("2024-01-24 04:08:41", utc=True)
    window_end = pd.to_datetime("2024-01-24 04:38:41", utc=True)
    candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

    # Filter to window
    df_tr_win = df_trace[(df_trace['ts'] >= window_start) & (df_trace['ts'] <= window_end)].copy()

    # Keep rows where trace_name contains 'to_ts-' or 'from_ts-'
    mask_dir = df_tr_win['trace_name'].astype(str).str.contains('to_ts-|from_ts-', regex=True, na=False)
    df_dir = df_tr_win[mask_dir].copy()

    # Parse target_service and trace_metric_type
    def parse_target_and_metric(name):
        if pd.isna(name):
            return pd.NA, pd.NA
        m = re.search(r'(?:to|from)_(ts-[^\.]+)(?:\.(.*))?', name)
        if not m:
            return pd.NA, pd.NA
        target = m.group(1)
        metric = m.group(2) if m.group(2) is not None else pd.NA
        return target, metric

    parsed = df_dir['trace_name'].apply(lambda x: pd.Series(parse_target_and_metric(x), index=['target_service','trace_metric_type']))
    df_dir = pd.concat([df_dir.reset_index(drop=True), parsed.reset_index(drop=True)], axis=1)

    # Select rows where source in candidates OR target in candidates
    mask_select = df_dir['cmdb_id'].isin(candidates) | df_dir['target_service'].isin(candidates)
    df_sel = df_dir[mask_select].copy()

    # Normalize trace_metric_type fallback
    df_sel['trace_metric_type'] = df_sel['trace_metric_type'].where(df_sel['trace_metric_type'].notna(),
                                                                    df_sel['trace_name'].astype(str).apply(lambda s: s.split('.')[-1] if '.' in s else pd.NA))
    df_sel['source_cmdb_id'] = df_sel['cmdb_id']
    # ensure ts is datetime
    df_sel['ts'] = pd.to_datetime(df_sel['ts'], utc=True)

# Now filter to edges where BOTH source and target are in candidates
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]
allowed_metrics = {'duration_mean','duration_p95','row_count','error_rate'}

mask_both = df_sel['source_cmdb_id'].isin(candidates) & df_sel['target_service'].isin(candidates)
df_inter = df_sel[mask_both].copy()

# If no inter-candidate edges, prepare explicit empty indicator
if df_inter.shape[0] == 0:
    inter_edges_res = pd.DataFrame(columns=[
        'source_cmdb_id','target_service','trace_metric_type',
        'count_in_window','first_ts_in_window','last_ts_in_window',
        'median_value_in_window','max_value_in_window'
    ])
    empty_indicator = "No inter-candidate edges found"
else:
    # Keep only allowed metric types
    df_inter = df_inter[df_inter['trace_metric_type'].isin(allowed_metrics)].copy()
    if df_inter.shape[0] == 0:
        inter_edges_res = pd.DataFrame(columns=[
            'source_cmdb_id','target_service','trace_metric_type',
            'count_in_window','first_ts_in_window','last_ts_in_window',
            'median_value_in_window','max_value_in_window'
        ])
        empty_indicator = "No inter-candidate edges found with allowed metric types"
    else:
        # Aggregate
        agg_inter = (
            df_inter
            .groupby(['source_cmdb_id','target_service','trace_metric_type'], dropna=False)['value']
            .agg(
                count_in_window='size',
                first_ts_in_window=lambda x: df_inter.loc[x.index, 'ts'].min(),
                last_ts_in_window=lambda x: df_inter.loc[x.index, 'ts'].max(),
                median_value_in_window='median',
                max_value_in_window='max'
            )
            .reset_index()
        )
        # Format timestamps
        agg_inter['first_ts_in_window'] = pd.to_datetime(agg_inter['first_ts_in_window'], utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
        agg_inter['last_ts_in_window']  = pd.to_datetime(agg_inter['last_ts_in_window'],  utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')

        # Reorder columns and sort
        inter_edges_res = agg_inter[[
            'source_cmdb_id','target_service','trace_metric_type',
            'count_in_window','first_ts_in_window','last_ts_in_window',
            'median_value_in_window','max_value_in_window'
        ]].sort_values(['count_in_window','median_value_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)

        empty_indicator = pd.NA

# Return result (compact)
inter_edges_res, empty_indicator

```
Out[7]:
```
Summary of inter-candidate trace-call edges (window 2024-01-24 04:08:41 → 04:38:41 UTC)

What was found
- Only self→self call edges among the five candidates were present (source == target). No cross-service edges between different candidate services were observed.
- All returned groups cover samples from 2024-01-24T04:11:00Z to 2024-01-24T04:35:00Z and have full coverage (count_in_window = 50).

Key aggregated values (top highlights)
- Row counts (traffic per service)
  - ts-route-service → ts-route-service: median row_count = 674, max = 786
  - ts-train-service → ts-train-service: median = 360, max = 480
  - ts-auth-service → ts-auth-service: median = 358, max = 400
  - ts-travel-service → ts-travel-service: median = 305, max = 425
  - ts-order-service → ts-order-service: median = 30, max = 67

- Durations (mean and p95)
  - ts-travel-service duration_p95: median ≈ 0.01419 s, max ≈ 0.01526 s; duration_mean median ≈ 0.00553 s
  - ts-order-service duration_p95: median ≈ 0.01318 s, max ≈ 0.38921 s — notable max spike above the median
  - ts-order-service duration_mean: median ≈ 0.00495 s, max ≈ 0.14452 s
  - ts-auth-service duration_mean median ≈ 0.00262 s, duration_p95 median ≈ 0.00188 s
  - ts-route-service and ts-train-service duration_mean/p95 medians are small (~0.001–0.003 s)

- Error rates
  - error_rate is zero across these self-edge groups (median and max = 0).

Concise interpretation
- Candidate services mainly produced internal/self traffic during the window; no direct candidate→candidate call edges were observed.
- ts-route-service shows the highest traffic volume (highest row_count median).
- ts-order-service shows the most notable duration spike (a max duration_p95 ≈ 0.389 s and duration_mean max ≈ 0.145 s) despite low median durations — this is the only prominent duration outlier among the inter-candidate self-edges.
- Error rates are not elevated (all zeros) for these self-call groups.

The original code execution output of IPython Kernel is also provided below for reference:

(       source_cmdb_id     target_service trace_metric_type  count_in_window    first_ts_in_window     last_ts_in_window  median_value_in_window  max_value_in_window
0    ts-route-service   ts-route-service         row_count               50  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z              674.000000           786.000000
1    ts-train-service   ts-train-service         row_count               50  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z              360.000000           480.000000
2     ts-auth-service    ts-auth-service         row_count               50  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z              358.000000           400.000000
3   ts-travel-service  ts-travel-service         row_count               50  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z              305.000000           425.000000
4    ts-order-service   ts-order-service         row_count               50  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z               30.000000            67.000000
5   ts-travel-service  ts-travel-service      duration_p95               50  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z                0.014186             0.015261
6    ts-order-service   ts-order-service      duration_p95               50  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z                0.013175             0.389206
7   ts-travel-service  ts-travel-service     duration_mean               50  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z                0.005533             0.005937
8    ts-order-service   ts-order-service     duration_mean               50  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z                0.004951             0.144521
9     ts-auth-service    ts-auth-service     duration_mean               50  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z                0.002620             0.005200
10    ts-auth-service    ts-auth-service      duration_p95               50  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z                0.001883             0.010382
11   ts-route-service   ts-route-service      duration_p95               50  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z                0.001704             0.001851
12   ts-train-service   ts-train-service      duration_p95               50  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z                0.001632             0.001857
13   ts-route-service   ts-route-service     duration_mean               50  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z                0.001282             0.001388
14   ts-train-service   ts-train-service     duration_mean               50  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z                0.001193             0.001476
15    ts-auth-service    ts-auth-service        error_rate               50  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z                0.000000             0.000000
16   ts-order-service   ts-order-service        error_rate               50  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z                0.000000             0.000000
17   ts-route-service   ts-route-service        error_rate               50  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z                0.000000             0.000000
18   ts-train-service   ts-train-service        error_rate               50  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z                0.000000             0.000000
19  ts-travel-service  ts-travel-service        error_rate               50  2024-01-24T04:11:00Z  2024-01-24T04:35:00Z                0.000000             0.000000, <NA>)```
```